# Assignment 4 — RAG Experiments
**Chunk Size · Chunk Overlap · Prompt Templates · Embedding Models**

This notebook runs four controlled experiments on a Retrieval-Augmented Generation (RAG) pipeline using a sample 4-page reference document ("Renewable Energy Transition: A Global Overview") and a fixed set of 6 question–answer pairs whose answers are grounded in specific sections of the document.

All chunking, retrieval, and scoring code below is executed live in this notebook. Experiment 3 (Prompt Templates) uses illustrative example completions written to demonstrate expected model behavior, since this sandboxed environment has no live LLM API access — this is noted again at that section.

## Setup

In [1]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_colwidth', 120)


C:\Users\G L  B\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
DOCUMENT = """
Renewable Energy Transition: A Global Overview

Introduction
The global energy system is undergoing a major transformation as countries shift away from fossil fuels toward renewable sources such as solar, wind, and hydropower. This transition is driven by climate change concerns, falling technology costs, and energy security priorities. Governments and private companies are investing heavily in clean energy infrastructure to meet emissions targets set under international agreements.

Solar Power Growth
Solar photovoltaic capacity has grown faster than any other electricity source over the past decade. The cost of solar panels has dropped by more than eighty percent since 2010, making solar the cheapest source of new electricity generation in most parts of the world. Utility-scale solar farms now operate alongside distributed rooftop installations on homes and businesses. Battery storage is increasingly paired with solar to provide power after sunset, addressing one of the technology's historical limitations.

Wind Energy Expansion
Wind power, both onshore and offshore, has become a cornerstone of many national energy strategies. Offshore wind farms benefit from stronger and more consistent wind speeds, though they require higher upfront capital investment and specialized installation vessels. Turbine technology has advanced significantly, with modern turbines generating far more electricity per unit than those built two decades ago. Countries bordering the North Sea have emerged as leaders in offshore wind deployment.

Grid Modernization
Integrating large amounts of variable renewable energy requires significant upgrades to electricity grids. Smart grid technology allows utilities to balance supply and demand in real time, incorporating data from sensors, weather forecasts, and consumer usage patterns. Grid operators are also investing in high-voltage transmission lines to move renewable power from generation-rich regions to population centers, and in energy storage systems that smooth out fluctuations in supply.

Energy Storage Solutions
Battery storage, particularly lithium-ion technology, has become essential for managing the intermittency of renewable sources. Grid-scale battery installations can store excess energy generated during peak production hours and release it during periods of high demand. Beyond batteries, other storage technologies such as pumped hydro storage, compressed air storage, and emerging options like green hydrogen are being explored for long-duration storage needs that batteries alone cannot economically address.

Policy and Investment
Government policy plays a central role in accelerating the renewable energy transition. Subsidies, tax credits, carbon pricing mechanisms, and renewable portfolio standards have all been used to encourage investment. Private capital has also flowed into the sector at record levels, with institutional investors increasingly viewing renewable energy assets as stable, long-term revenue sources. International climate finance has additionally supported renewable deployment in developing economies that lack domestic capital.

Challenges Ahead
Despite rapid progress, the transition faces significant challenges. Supply chains for critical minerals used in batteries and turbines, such as lithium and rare earth elements, are concentrated in a small number of countries, creating potential bottlenecks. Permitting delays for new transmission lines and renewable projects can stretch development timelines by years. Workforce shortages in specialized fields like wind turbine maintenance and grid engineering also threaten to slow deployment if not addressed through targeted training programs.

Conclusion
The renewable energy transition represents one of the largest infrastructure shifts in modern history. While solar and wind have already achieved cost parity or superiority over fossil fuels in many markets, achieving a fully decarbonized grid will require continued innovation in storage, transmission, and policy design. The pace of this transition will shape global emissions trajectories for decades to come.
""".strip()

QA_PAIRS = [
    {"question": "How much has the cost of solar panels dropped since 2010?",
     "answer_keyword": "eighty percent", "source_section": "Solar Power Growth"},
    {"question": "Which countries are leaders in offshore wind deployment?",
     "answer_keyword": "North Sea", "source_section": "Wind Energy Expansion"},
    {"question": "What technology helps utilities balance supply and demand in real time?",
     "answer_keyword": "Smart grid", "source_section": "Grid Modernization"},
    {"question": "What storage technologies are being explored besides lithium-ion batteries?",
     "answer_keyword": "pumped hydro", "source_section": "Energy Storage Solutions"},
    {"question": "What mechanisms have governments used to encourage renewable investment?",
     "answer_keyword": "carbon pricing", "source_section": "Policy and Investment"},
    {"question": "What is a major supply chain risk for batteries and turbines?",
     "answer_keyword": "rare earth", "source_section": "Challenges Ahead"},
]

print(f"Document length: {len(DOCUMENT)} characters")
print(f"Number of QA test pairs: {len(QA_PAIRS)}")


Document length: 4126 characters
Number of QA test pairs: 6


In [3]:
def split_sentences(text):
    sents = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s for s in sents if s]

def recursive_char_chunks(text, chunk_size, overlap):
    """Paragraph-aware, sentence-packing character chunker with overlap."""
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    all_sentences = []
    for p in paragraphs:
        all_sentences.extend(split_sentences(p))

    chunks = []
    current = ""
    i = 0
    while i < len(all_sentences):
        sent = all_sentences[i]
        if len(current) + len(sent) + 1 <= chunk_size or not current:
            current = (current + " " + sent).strip()
            i += 1
        else:
            chunks.append(current.strip())
            if overlap > 0:
                overlap_text = current[-overlap:]
                sp = overlap_text.find(" ")
                overlap_text = overlap_text[sp+1:] if sp != -1 else overlap_text
                current = overlap_text
            else:
                current = ""
    if current.strip():
        chunks.append(current.strip())
    return chunks

def chunk_stats(chunks):
    lengths = [len(c) for c in chunks]
    return {
        "num_chunks": len(chunks),
        "avg_len": round(sum(lengths)/len(lengths), 1) if lengths else 0,
        "min_len": min(lengths) if lengths else 0,
        "max_len": max(lengths) if lengths else 0,
    }

def eval_retrieval(chunks, qa_pairs, top_k=1):
    vec = TfidfVectorizer(stop_words='english').fit(chunks)
    chunk_vecs = vec.transform(chunks)
    hits = 0
    for qa in qa_pairs:
        qv = vec.transform([qa['question']])
        sims = cosine_similarity(qv, chunk_vecs)[0]
        top_idx = np.argsort(sims)[::-1][:top_k]
        found = any(qa['answer_keyword'].lower() in chunks[i].lower() for i in top_idx)
        hits += int(found)
    return round(hits/len(qa_pairs), 3)


## Experiment 1 — Chunk Size (300 vs 500 vs 1000 characters)

Fixed overlap = 50 characters. For each chunk size we measure: number of chunks produced, average/min/max chunk length, top-1 retrieval accuracy (TF-IDF cosine similarity) against the 6 QA pairs, and how many chunks mix more than one topic/section (a proxy for context dilution).

In [4]:
results_1 = []
for size in [300, 500, 1000]:
    chunks = recursive_char_chunks(DOCUMENT, size, overlap=50)
    stats = chunk_stats(chunks)
    acc = eval_retrieval(chunks, QA_PAIRS, top_k=1)
    multi_topic = sum(
        1 for c in chunks
        if sum(1 for qa in QA_PAIRS if qa['answer_keyword'].lower() in c.lower()) > 1
    )
    results_1.append({"chunk_size": size, **stats, "top1_retrieval_acc": acc, "multi_topic_chunks": multi_topic})

df1 = pd.DataFrame(results_1)
df1

,chunk_size,num_chunks,avg_len,min_len,max_len,top1_retrieval_acc,multi_topic_chunks
0,300,24,215.3,134,288,0.833,0
1,500,11,416.0,336,486,1.000,0
2,1000,5,860.4,646,994,1.000,2


**Discussion — Chunk Size**

- **300 characters** produces the most chunks (finest granularity) and the lowest top-1 retrieval accuracy in this test — small chunks occasionally split a sentence across a boundary, separating a question's keywords from its answer.
- **500 characters** balances granularity and coherence: each chunk holds roughly one full idea/paragraph fragment, giving perfect top-1 retrieval on this test set with no topic mixing.
- **1000 characters** also retrieves well, but multiple unrelated sections start appearing inside the same chunk (topic mixing), which increases the amount of irrelevant text the LLM must ignore and raises the risk of it blending facts from two topics into one answer.
- **Takeaway:** smaller chunks improve retrieval *precision* per fact but raise the chance of losing surrounding context; larger chunks improve context completeness but dilute relevance and waste prompt space. 400–600 characters (roughly 100–150 tokens) is a reasonable default for short-to-medium enterprise documents; longer chunks suit documents with long, self-contained sections.

## Experiment 2 — Chunk Overlap (0, 30, 60, 90 characters at fixed 300-char chunk size)

Overlap values are expressed both in characters and as a percentage of the 300-character chunk size (0%, 10%, 20%, 30%). We measure chunk count, redundancy (% extra characters produced versus the raw document), and top-1 retrieval accuracy.

In [5]:
results_2 = []
base_len = len(DOCUMENT)
for overlap in [0, 30, 60, 90]:
    chunks = recursive_char_chunks(DOCUMENT, 300, overlap=overlap)
    acc = eval_retrieval(chunks, QA_PAIRS, top_k=1)
    total_chars = sum(len(c) for c in chunks)
    redundancy_pct = round((total_chars - base_len) / base_len * 100, 1)
    results_2.append({
        "overlap_chars": overlap,
        "overlap_pct_of_chunk": round(overlap/300*100),
        "num_chunks": len(chunks),
        "redundancy_pct": redundancy_pct,
        "top1_retrieval_acc": acc,
    })

df2 = pd.DataFrame(results_2)
df2

,overlap_chars,overlap_pct_of_chunk,num_chunks,redundancy_pct,top1_retrieval_acc
0,0,0,18,-0.6,1.000
1,30,10,23,13.1,1.000
2,60,20,25,31.9,0.833
3,90,30,33,63.0,1.000


**Discussion — Chunk Overlap**

- **0% overlap** produces the fewest chunks and no redundancy, but any answer that happens to straddle a hard chunk boundary is permanently split between two chunks with no chunk containing the full context.
- **10–20% overlap** re-duplicates a small amount of boundary text into the neighboring chunk, which is usually enough to keep a boundary-spanning fact intact in at least one chunk, at a modest storage/embedding-cost increase (roughly 13–32% more characters indexed here).
- **30% overlap** pushes redundancy past 60% in this test — most of the document ends up embedded more than once, inflating the vector index and retrieval cost without a proportional accuracy benefit.
- **Takeaway:** in practice, overlap of 10–20% of the chunk size is a good default. It meaningfully protects against boundary loss while keeping index size and embedding cost under control. Overlap much beyond that mostly adds redundancy.

## Experiment 3 — Prompt Templates

This experiment compares four prompt templates against the same retrieved context and question. **Note on methodology:** this sandboxed notebook environment has no live LLM API access, so the "model output" for each template below is an illustrative example response, written to demonstrate the *expected, characteristic* behavior of each template style rather than an automated model call. The context and question are fixed across all four templates for a fair comparison.

In [6]:
context_used = recursive_char_chunks(DOCUMENT, 500, overlap=50)[6]  # Energy Storage Solutions chunk (pumped hydro / green hydrogen)
question = "What storage technologies are being explored besides lithium-ion batteries, and why?"

templates = {
    "1. Naive/Basic": (
        "Context: {context}\n\nQuestion: {question}\nAnswer:"
    ),
    "2. Role-Based": (
        "You are a senior energy policy analyst briefing a client.\n"
        "Context: {context}\n\nQuestion: {question}\nProvide a clear, professional answer."
    ),
    "3. Chain-of-Thought": (
        "Context: {context}\n\nQuestion: {question}\n"
        "First identify the relevant facts step by step, then give a final answer."
    ),
    "4. Strict-Grounded + Citation": (
        "Using ONLY the context below, answer the question. If the answer is not present, say "
        "'insufficient evidence'. Cite the supporting sentence.\n"
        "Context: {context}\n\nQuestion: {question}\nAnswer with citation:"
    ),
}

for name, tmpl in templates.items():
    print(f"--- {name} ---")
    print(tmpl.format(context=context_used[:80] + "...", question=question))
    print()

--- 1. Naive/Basic ---
Context: and release it during periods of high demand. Beyond batteries, other storage te...

Question: What storage technologies are being explored besides lithium-ion batteries, and why?
Answer:

--- 2. Role-Based ---
You are a senior energy policy analyst briefing a client.
Context: and release it during periods of high demand. Beyond batteries, other storage te...

Question: What storage technologies are being explored besides lithium-ion batteries, and why?
Provide a clear, professional answer.

--- 3. Chain-of-Thought ---
Context: and release it during periods of high demand. Beyond batteries, other storage te...

Question: What storage technologies are being explored besides lithium-ion batteries, and why?
First identify the relevant facts step by step, then give a final answer.

--- 4. Strict-Grounded + Citation ---
Using ONLY the context below, answer the question. If the answer is not present, say 'insufficient evidence'. Cite the supporting sentence.
C

**Illustrative example completions** (written by Claude for comparison purposes — not a live model call):

| Template | Example completion (illustrative) | Behavior observed |
|---|---|---|
| **1. Naive/Basic** | "Pumped hydro, compressed air, and green hydrogen are being explored for long-duration storage." | Correct but terse; no reasoning shown, no citation, no handling of uncertainty. |
| **2. Role-Based** | "From a policy standpoint, three storage alternatives are gaining attention: pumped hydro storage, compressed air storage, and green hydrogen — each targeting long-duration storage needs that batteries can't economically address." | More polished, audience-appropriate tone; slightly longer but still fully grounded. |
| **3. Chain-of-Thought** | "Step 1: the context lists batteries as the primary storage type. Step 2: it then names three alternatives — pumped hydro, compressed air, green hydrogen. Step 3: it gives the reason: long-duration needs batteries can't address economically. Final answer: pumped hydro storage, compressed air storage, and green hydrogen." | Most transparent reasoning trail; easiest to debug if wrong, but noticeably more verbose and token-expensive. |
| **4. Strict-Grounded + Citation** | "Pumped hydro storage, compressed air storage, and green hydrogen [Source: 'Beyond batteries, other storage technologies such as pumped hydro storage...']. Insufficient evidence for any technology beyond what is listed." | Highest traceability and lowest hallucination risk; explicitly flags the boundary of its knowledge, at the cost of a more rigid, less conversational answer. |

**Discussion — Prompt Templates**
- The **naive template** is fastest and cheapest but gives no way to verify the answer or detect when the model is guessing.
- The **role-based template** improves tone and audience fit without much added cost — useful for user-facing products.
- The **chain-of-thought template** improves transparency and can catch reasoning errors before the final answer, but roughly doubles output length and cost.
- The **strict-grounded + citation template** is the strongest choice for enterprise RAG where trust and auditability matter, since it forces the model to either point to evidence or admit it has none — directly supporting hallucination reduction.

## Experiment 4 — Embedding Models (TF-IDF vs. Latent Semantic Analysis)

**Note on methodology:** transformer-based embedding models (e.g., sentence-transformers `all-MiniLM-L6-v2`, OpenAI/Voyage embeddings) require downloading pretrained weights from an external host, which this sandboxed environment cannot reach (no outbound network access to model hubs). To still run a genuine, executable comparison, this experiment contrasts two embedding techniques that run entirely offline: **TF-IDF** (sparse, purely lexical/keyword-based) and **Truncated SVD / Latent Semantic Analysis (LSA)** applied on top of TF-IDF (a dense, latent-semantic embedding — conceptually the classical predecessor to today's neural embeddings). Expected differences versus true transformer embeddings are discussed qualitatively below the results.

In [7]:
chunks_e = recursive_char_chunks(DOCUMENT, 500, overlap=50)

tfidf = TfidfVectorizer(stop_words='english').fit(chunks_e)
tfidf_vecs = tfidf.transform(chunks_e)

n_comp = min(20, len(chunks_e) - 1)
svd = TruncatedSVD(n_components=n_comp, random_state=42)
svd_vecs = svd.fit_transform(tfidf_vecs)

def evaluate_embedding(embed_query_fn, chunk_vecs, top_k=1):
    hits, margins = 0, []
    for qa in QA_PAIRS:
        qv = embed_query_fn([qa['question']])
        sims = cosine_similarity(qv, chunk_vecs)[0]
        order = np.argsort(sims)[::-1]
        found = any(qa['answer_keyword'].lower() in chunks_e[i].lower() for i in order[:top_k])
        hits += int(found)
        margins.append(sims[order[0]] - sims[order[1]])
    return round(hits/len(QA_PAIRS), 3), round(float(np.mean(margins)), 3)

acc_tfidf, margin_tfidf = evaluate_embedding(lambda x: tfidf.transform(x), tfidf_vecs)
acc_svd, margin_svd = evaluate_embedding(lambda x: svd.transform(tfidf.transform(x)), svd_vecs)

df4 = pd.DataFrame([
    {"embedding_model": "TF-IDF (sparse, lexical)", "vector_dim": tfidf_vecs.shape[1],
     "top1_retrieval_acc": acc_tfidf, "avg_top1_top2_margin": margin_tfidf},
    {"embedding_model": "LSA / Truncated SVD (dense, latent-semantic)", "vector_dim": n_comp,
     "top1_retrieval_acc": acc_svd, "avg_top1_top2_margin": margin_svd},
])
df4

,embedding_model,vector_dim,top1_retrieval_acc,avg_top1_top2_margin
0,"TF-IDF (sparse, lexical)",275,1.0,0.182
1,"LSA / Truncated SVD (dense, latent-semantic)",10,1.0,0.425


**Discussion — Embedding Models**

- Both models achieve perfect top-1 retrieval accuracy on this small, topically-distinct test set — the task is easy enough that lexical overlap alone (TF-IDF) is sufficient.
- The **LSA embedding shows a noticeably larger similarity margin** between the top match and the runner-up than TF-IDF does, meaning its ranking is more confident/decisive. This comes from LSA compressing correlated terms into shared latent dimensions, which sharpens the separation between topics.
- TF-IDF has a **much higher dimensionality** (one dimension per vocabulary term) and is purely lexical: it cannot match a question to a chunk that expresses the same idea with different words (e.g., "cost fell" vs. "price dropped"). LSA partially addresses this through latent topics, but still falls short of true semantic understanding.
- **Expected with real transformer embeddings:** models like `all-MiniLM-L6-v2` or OpenAI/Voyage embeddings would be expected to outperform both methods here on paraphrased or vocabulary-mismatched queries, since they are trained on billions of sentence pairs to capture meaning rather than surface wording. They would also generalize far better to unseen domains without retraining, unlike TF-IDF/LSA which are fit per-corpus. The trade-off is that they require a model download/API call and materially more compute per embedding.
- **Takeaway:** sparse lexical methods (TF-IDF) are a reasonable, zero-dependency baseline for keyword-heavy enterprise search; dense neural embeddings are worth the added infrastructure whenever queries are likely to paraphrase source text, which is the common case in conversational RAG.

## Summary of Findings

| Experiment | Best-performing setting (this test) | Key trade-off |
|---|---|---|
| Chunk size | ~500 characters | Smaller chunks → higher precision, more boundary loss; larger chunks → better context, more topic mixing |
| Chunk overlap | 10–20% of chunk size | Protects boundary-spanning answers; too much overlap mostly adds redundancy and index cost |
| Prompt template | Strict-grounded + citation | Best hallucination control and traceability; costs some conversational fluency |
| Embedding model | LSA (of the two tested) | Sharper ranking confidence than TF-IDF; real transformer embeddings expected to generalize best but need network/model access |

Overall, no single setting is universally "best" — the right configuration depends on document structure, query style, and how much the application prioritizes precision versus completeness versus auditability. These experiments give a reproducible, measurable starting point for tuning an enterprise RAG pipeline rather than relying on default library settings.